# Lab 6: A Table for Any Formula

**MATH 170 — Introduction to Scientific Computing**
**Fall 2026**

Your made-up lab partner in physics keeps asking you for tables of values: the
height of a ball thrown upward, then a cooling cup of coffee, then a spring.
Every time, you edit the formula inside your program and run it again.

This lab builds the program that ends that: you type the formula, the interval
and the number of steps **after the program's name**, and it prints the table.

```text
Terminal> python tabulate.py "20*x - 4.9*x**2" 0 4 8
```

And because a program you hand to someone else had better work, you also write
the **tests** that check it.

Everything comes from **§4.7, §5.1 and §5.2**: test functions with `assert`,
comparing floats with a tolerance, `pytest`, command line arguments in
`sys.argv`, and turning a string into code with `eval`.

---

## Before anything else

Run this cell first. It makes `sin`, `cos`, `exp`, `sqrt`, `pi` and the rest of
`math` available, so a formula like `"sin(x) + 1"` can use them.

In [ ]:
# Just run this one.
from math import *
import sys

print("ready -- sin(pi/2) =", sin(pi/2))

---

## What you will do

**Tasks 1–4** — four short functions. Work each one out here in the notebook,
where you can run it and see what it does. At the end, one cell collects your
four functions and writes them into **`lab06_table.py`** for you.

That file is the one that gets graded. It travels: your laptop → your own
repository on GitHub (`git push`) → Gradescope, which reads it from there.

- **Tasks 1–3** build the program: read the command line, turn the formula into
  a function, make the table.
- **Task 4** is a **test function** that checks Task 2. You write the whole
  function, `def` line included, using exactly the name the task gives.

Then you run your work two ways a real program is run: with `pytest`, and from
the command line with `tabulate.py`.

**Grading is on completion.** Each of the four tasks is worth **25 points**,
earned by doing real work on it — your answer does not have to be right. The
autograder still shows you which checks passed, so use it to find what to fix.

---

## Task 1 — Read the command line (§5.1)

When someone types

```text
Terminal> python tabulate.py "20*x - 4.9*x**2" 0 4 8
```

the program `tabulate.py` receives the list

```python
sys.argv == ['tabulate.py', '20*x - 4.9*x**2', '0', '4', '8']
```

`parse_args(argv)` takes a list like that and returns **four** values, in this
order: the formula (a string), the left end `a` and the right end `b` (floats),
and the number of steps `n` (an `int` — it counts something).

You never type `sys.argv` here. The test hands `parse_args` a list that looks
exactly like it, so you can try it without a terminal.

In [ ]:
def parse_args(argv):
    """
    The formula, a, b and n from a command line, returned as four values.

    argv : a list like sys.argv, e.g. ['tabulate.py', 'x**2', '0', '2', '4']
           -> 'x**2', 0.0, 2.0, 4
    """
    # TODO: your code here
    return None

In [ ]:
# TEST -- Task 1
result = parse_args(['tabulate.py', '20*x - 4.9*x**2', '0', '4', '8'])
expected = ('20*x - 4.9*x**2', 0.0, 4.0, 8)
print("got:     ", result)
print("expected:", expected)

if result is None:
    print("Not implemented yet -- the function still returns None.")
elif not isinstance(result, tuple) or len(result) != 4:
    print("Return four values, separated by commas: return formula, a, b, n")
elif result[0] == 'tabulate.py':
    print("The formula came out as 'tabulate.py'. Which element of argv is the")
    print("program's own name?")
elif result[1:] == ('0', '4', '8'):
    print("a, b and n are still strings. Everything in sys.argv is a string.")
elif result == expected and type(result[1]) == float and type(result[3]) == int:
    print("PASSED")
elif type(result[3]) != int:
    print("n counts steps, so it should be an int, not", type(result[3]).__name__)
else:
    print("Not matching -- print argv[1], argv[2], ... and compare.")

---

## Task 2 — From a formula to a function (§5.2)

`make_function(formula)` takes a formula in `x`, as a string, and returns a
**function** of `x` that evaluates it:

```python
f = make_function('x**2 + 1')
f(3)          # 10
```

The book does this by writing a whole `def` into a string and running it with
`exec`. There is a shorter way: a `lambda` (§4.5) is an *expression*, and `eval`
turns a string holding an expression into its value. Build the right string,
and its value is the function you want.

Its body is one line.

In [ ]:
def make_function(formula):
    """
    A function of x that evaluates formula.

    formula : a string with a formula in x, e.g. 'x**2 + 1' or 'sin(x)'
    """
    # TODO: your code here
    return None

In [ ]:
# TEST -- Task 2
f = make_function('x**2 + 1')

if f is None:
    print("Not implemented yet -- the function still returns None.")
elif not callable(f):
    print("make_function returned", repr(f), "-- a value, not a function.")
    print("The string you hand to eval has to be a whole lambda.")
else:
    g = make_function('sin(x)')
    print("make_function('x**2 + 1')(3)  =", f(3), "   expected 10")
    print("make_function('sin(x)')(pi/2) =", g(pi/2), "   expected 1.0")
    if f(3) == 10 and abs(g(pi/2) - 1.0) < 1e-12:
        print("PASSED")
    else:
        print("Not matching -- print the string you pass to eval and read it.")

If the test above stops with `NameError: name 'x' is not defined`, `eval` tried
to compute the formula *right now* — and there is no `x` yet. You want it to
build a function that waits for an `x`.

---

## Task 3 — The table (§3, with a function handed in)

`table(f, a, b, n)` splits the interval $[a, b]$ into `n` equal steps of size
$h = (b - a)/n$ and returns **two lists**: the points

$$x_k = a + k\,h, \qquad k = 0, 1, \dots, n,$$

and the values $f(x_k)$. Both ends are included, so there are `n + 1` points,
not `n`.

In [ ]:
def table(f, a, b, n):
    """
    The points x_0, ..., x_n splitting [a, b] into n equal steps, and the
    values f(x_k) at them, returned as two lists: xs, ys.

    f    : a function of one number
    a, b : the ends of the interval
    n    : the number of steps (there are n + 1 points)
    """
    # TODO: your code here
    return None

In [ ]:
# TEST -- Task 3
result = table(lambda x: x**2, 0, 2, 4)
print("table(lambda x: x**2, 0, 2, 4) =", result)
print("expected                       = ([0.0, 0.5, 1.0, 1.5, 2.0], [0.0, 0.25, 1.0, 2.25, 4.0])")

if result is None:
    print("Not implemented yet -- the function still returns None.")
elif not isinstance(result, tuple) or len(result) != 2:
    print("Return the two lists separated by a comma: return xs, ys")
else:
    xs, ys = result
    if len(xs) == 4:
        print("Four points, not five -- the right end is missing.")
        print("range(n) stops at n - 1. How many points do you need?")
    elif xs == [0.0, 0.5, 1.0, 1.5, 2.0] and ys == [0.0, 0.25, 1.0, 2.25, 4.0]:
        print("PASSED")
    elif xs == [0.0, 0.5, 1.0, 1.5, 2.0]:
        print("The points are right, the values are not. Call f on each point.")
    else:
        print("Not matching -- print h, then each x_k as you compute it.")

### Your table

Run the cell below and type a formula in `x` when it asks — for example the
ball, `20*x - 4.9*x**2`. It uses `input`, so whatever you type arrives as a
string (§5.1). Here that is exactly what `make_function` wants.

In [ ]:
# Just run this one.
ready = (parse_args(['p', 'x', '0', '1', '1']) is not None
         and make_function('x') is not None
         and table(lambda x: x, 0, 1, 1) is not None)

if not ready:
    print("Finish Tasks 1-3, run their cells, then run this one again.")
else:
    formula = input("A formula in x, for example 20*x - 4.9*x**2 : ")
    f = make_function(formula)
    xs, ys = table(f, 0, 4, 8)
    print()
    print("       x      ", formula)
    for k in range(len(xs)):
        print(f"{xs[k]:8.2f}   {ys[k]:12.5f}")

---

## Task 4 — Test `make_function` (§4.7)

Write a test function **`test_make_function()`** — the whole thing, starting
with `def`. Follow the rules for test functions from §4.7:

- it takes **no arguments**;
- it checks **at least two formulas** whose values you can work out by hand,
  at least one of them using something from `math` — `sin`, `exp`, `sqrt`, ...;
- it compares floats with a **tolerance**, never with `==`;
- every `assert` has a message saying what went wrong.

A test that passes prints nothing. The TEST cell below runs yours against your
own `make_function`, and then against two broken ones. A good test passes the
first and catches both of the others.

In [ ]:
# TODO: your code here

In [ ]:
# TEST -- Task 4. Just run it.
# This one is a tool, not an exercise -- you are not expected to read it.
def run_against(test, name, fake):
    # Swap in a fake for name, run the test, put the real one back.
    # True if the test failed (an AssertionError, or any error the fake caused).
    real = globals()[name]
    globals()[name] = fake
    try:
        test()
        return False
    except Exception:
        return True
    finally:
        globals()[name] = real

if "test_make_function" not in globals():
    print("No function called test_make_function yet -- write it above and run that cell.")
elif make_function('x') is None:
    print("Finish Task 2 first -- your test needs a make_function to test.")
else:
    test_make_function()
    print("1. Your make_function:                  test passed (silently, as it should)")

    caught = run_against(test_make_function, "make_function",
                         lambda formula: (lambda x: 0))
    print("2. A make_function that always gives 0: ", "CAUGHT" if caught else "missed")

    caught2 = run_against(test_make_function, "make_function",
                          lambda formula: (lambda x: x))
    print("3. A make_function that ignores the formula:", "CAUGHT" if caught2 else "missed")

    if caught and caught2:
        print("PASSED")
    else:
        print("Your test let a broken make_function through. Which values does it check?")

If line 1 stops with an `AssertionError`, read its message: your test says your
`make_function` is wrong. Either the function or the test has a mistake — work
out a value by hand to decide which.

---

## Submitting

### Step 1 — Build `lab06_table.py` from your work

The cell below collects the four functions you wrote above and writes them into
`lab06_table.py`, which sits in this same folder.

**Run every task cell first**, from the top, so that all four functions exist.
Then run this.

Change your mind about an answer later? Fix it in the task cell, run that cell,
and run this one again — the file is rewritten from scratch each time.

In [ ]:
# Run this to build lab06_table.py from the functions you wrote above.
# This one is a tool, not an exercise -- you are not expected to read it.
import ast
import inspect
import json
import os

FUNCTIONS = ["parse_args", "make_function", "table", "test_make_function"]
OUTFILE = "lab06_table.py"
NOTEBOOK = "lab06.ipynb"

HEADER = '''"""
MATH 170 - Lab 6: A Table for Any Formula
Fall 2026

Built from lab06.ipynb by the build cell at the end of the notebook.
To change something: edit the task cell, run it, and run the build cell again.
"""

import sys
from math import *

'''


def from_memory(name):
    # The function as you last ran it in this notebook.
    function = globals().get(name)
    if function is None:
        return None
    try:
        return inspect.getsource(function)
    except (OSError, TypeError):
        return None


def from_notebook(name):
    # Fallback: dig the function out of the saved .ipynb file.
    if not os.path.exists(NOTEBOOK):
        return None
    with open(NOTEBOOK) as f:
        cells = json.load(f)["cells"]
    latest = None
    for cell in cells:
        if cell["cell_type"] != "code":
            continue
        source = "".join(cell["source"])
        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node, ast.FunctionDef) and node.name == name:
                latest = ast.get_source_segment(source, node)
    return latest


pieces = []
report = []
finished = 0
for name in FUNCTIONS:
    code = from_memory(name)
    if code is None:
        code = from_notebook(name)
    if code is None:
        report.append("  MISSING    " + name + " -- write it (spelled exactly like this) and run its cell")
    elif "TODO: your code here" in code:
        pieces.append(code.rstrip() + "\n")
        report.append("  unfinished " + name + " -- still the placeholder")
    else:
        pieces.append(code.rstrip() + "\n")
        report.append("  written    " + name)
        finished = finished + 1

print("\n".join(report))
print()

if finished == 0:
    print("Nothing to build yet -- none of the four are filled in.")
    print("Work through the tasks above, run their cells, then run this one again.")
else:
    with open(OUTFILE, "w") as f:
        f.write(HEADER + "\n\n".join(pieces))
    print("Wrote " + OUTFILE + " in " + os.getcwd())
    print("Now run the check below.")

### Step 2 — Check the file before you push

In [ ]:
# Run this. It imports YOUR lab06_table.py and checks it.
import importlib
import lab06_table
importlib.reload(lab06_table)

print("parse_args(['tabulate.py', 'x**2', '0', '2', '4']) ->",
      lab06_table.parse_args(['tabulate.py', 'x**2', '0', '2', '4']))
print("                                       should be  ('x**2', 0.0, 2.0, 4)")
print()
print("make_function('x**2 + 1')(3) ->", lab06_table.make_function('x**2 + 1')(3))
print("                    should be   10")
print()
print("table(lambda x: 2*x, 0, 1, 2) ->", lab06_table.table(lambda x: 2*x, 0, 1, 2))
print("                    should be   ([0.0, 0.5, 1.0], [0.0, 1.0, 2.0])")
print()
print("test_make_function is in the file:", hasattr(lab06_table, "test_make_function"))
print()
print("Anything printing None or False was not copied over. Anything else -- go on.")

### Step 3 — Run it the ways real programs run

**Your test, with `pytest`.** `pytest` finds every function in the file whose
name starts with `test_` and runs it. Predict the summary line, then run.

In [ ]:
!"{sys.executable}" -m pytest lab06_table.py

**Your program, from the command line.** `tabulate.py` came with the lab — you
do not edit it. It reads `sys.argv`, and hands it to *your* `parse_args`,
`make_function` and `table` from `lab06_table.py`. The line below is the same as
typing `python tabulate.py "20*x - 4.9*x**2" 0 4 8` in a terminal.

In [ ]:
!"{sys.executable}" tabulate.py "20*x - 4.9*x**2" 0 4 8

Is the ball still in the air after 4 seconds? Change the numbers on the line
above — not the program — until you can say roughly when it lands.

Now the same line **without the quotes**. Predict what `tabulate.py` receives in
`sys.argv`, then run it.

In [ ]:
!"{sys.executable}" tabulate.py 20*x - 4.9*x**2 0 4 8

Finally, open a real terminal — Terminal on a Mac, **Anaconda Prompt** on
Windows — `cd` into your `lab06` folder, and run a table of your own:

```text
python tabulate.py "sin(x)" 0 3.14159 6
```

(On a Mac, a formula with `*` and no quotes may not even reach Python: the
terminal answers `zsh: no matches found`. Quote the formula, always.)

### Step 4 — Push your work to your own repository

From inside your `math170-yourname` folder:

```
git status
git add lab06/lab06_table.py
git commit -m "Complete Lab 6"
git push
```

`git status` first, so you see what changed before you send anything. Then stage
the one file Gradescope needs, by name. Your notebook stays on your laptop —
and so does the `.pytest_cache` folder that `pytest` just made.

**Do not use `git add .`** It stages everything the folder happens to contain
that day. Once it is pushed it is in your repository's history, and taking it
back out is far more work than typing one filename.

This sends the file to `origin` — **your** repository. (`upstream` is the course
repository. You pull labs from it; you never push to it.)

Refresh your repository page on GitHub and check that `lab06/lab06_table.py` is
there, with your code in it. Gradescope reads your repository, so if the file is
not on GitHub there is nothing for it to grade.

**Pushing does not submit the lab.** It backs your work up. Submission is next.

### Step 5 — Submit on Gradescope

Open **gradescope.com directly in your browser** — **not through Canvas**, or
the GitHub connection will be refused.

Open **MATH 170 → Lab 06** and click **Submit**, then:

1. Choose **GitHub**
2. Pick your repository `math170-yourname` and the `main` branch
3. Submit

Gradescope takes a **snapshot** of your repository at that moment. If you push
more work afterwards, submit again — pushing alone changes nothing on
Gradescope.

If GitHub will not connect, upload `lab06_table.py` directly instead. That
works just as well.

Wait for the autograder to finish. You should see **100 / 100**. You may
resubmit as many times as you like; the last one counts.

---

## Before you leave

- [ ] All four tasks written, and their TEST cells run
- [ ] Ran the build cell, and the check in Step 2 printed no `None`
- [ ] Ran `pytest` and `tabulate.py`
- [ ] Committed and pushed — `lab06/lab06_table.py` visible in your repository on GitHub
- [ ] Gradescope shows **100 / 100** for Lab 06
- [ ] You signed the attendance sheet

Remember: **completion**, not correctness. A red check costs you nothing — the
25 points for a task are for working on it, not for getting it right. If you are
out of time, submit what you have.